# PRD Final Eval KG

Apple-to-apple final reevaluation for Baseline, Phase A, Phase B, and Phase C as specified in `PRD.md`.

Settings: `eval_split=testmini`, headline subset `eval_overall_numeric`, `max_eval_examples_per_subset=4`, `num_samples_per_prompt=1`, `max_completion_length=64`, `hardware_profile=kaggle_t4`, LoRA rank 8.

In [ ]:
import os, json, glob, shutil, subprocess, tarfile, base64, io, csv
from datetime import datetime

WORKDIR = "/kaggle/working/RL_GSPO_Qwen2.5VLM"
VENV_DIR = "/tmp/rl-gspo-venv"
VENV_PYTHON = f"{VENV_DIR}/bin/python"
OUTPUT_ROOT = "outputs_prd_final_eval_kg"
PROGRESS_PATH = "/kaggle/working/prd_final_eval_progress.txt"
CODE_ARCHIVE_B64 = """..."""

def write_progress(message):
    line = f"{datetime.utcnow().isoformat()} {message}"
    print(line, flush=True)
    with open(PROGRESS_PATH, "a", encoding="utf-8") as handle:
        handle.write(line + "\n")

write_progress("PRD final eval notebook start")
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_ARCHIVE_B64)), mode="r:gz") as tf:
    tf.extractall(WORKDIR)
write_progress("Extracted embedded evaluation code")
print("Workdir files:", sorted(os.listdir(WORKDIR)))


In [ ]:
def resolve_model_path():
    candidates = glob.glob("/kaggle/input/**/config.json", recursive=True)
    matches = []
    for config_path in candidates:
        root = os.path.dirname(config_path)
        if os.path.exists(os.path.join(root, "model.safetensors")) or glob.glob(os.path.join(root, "model-*.safetensors")):
            matches.append(root)
    if not matches:
        raise FileNotFoundError(f"Could not find model config.json + safetensors under /kaggle/input. Visible inputs: {glob.glob('/kaggle/input/*')}")
    matches = sorted(set(matches), key=len)
    if len(matches) > 1:
        write_progress(f"Multiple model candidates found: {matches}")
    return matches[0]

def find_selected_bundle():
    tar_matches = glob.glob("/kaggle/input/**/prd_final_eval_inputs_kg.tar.gz", recursive=True)
    if tar_matches:
        return ("tar", sorted(tar_matches)[0])
    for root, dirs, files in os.walk("/kaggle/input"):
        if "selected_checkpoint_manifest.json" in files and os.path.isdir(os.path.join(root, "selected_checkpoints")):
            return ("dir", root)
    raise FileNotFoundError(f"Could not find PRD final eval input bundle. Visible inputs: {glob.glob('/kaggle/input/*')}")

MODEL_PATH = resolve_model_path()
kind, bundle_path = find_selected_bundle()
INPUT_ROOT = os.path.join(WORKDIR, "prd_final_eval_inputs_kg")
if os.path.exists(INPUT_ROOT):
    shutil.rmtree(INPUT_ROOT)
if kind == "tar":
    with tarfile.open(bundle_path, "r:gz") as tf:
        tf.extractall(WORKDIR)
else:
    shutil.copytree(bundle_path, INPUT_ROOT)
write_progress(f"Resolved model path: {MODEL_PATH}")
write_progress(f"Resolved selected checkpoint input: {bundle_path}")
with open(os.path.join(INPUT_ROOT, "selected_checkpoint_manifest.json"), "r", encoding="utf-8") as handle:
    SELECTED_MANIFEST = json.load(handle)
print(json.dumps(SELECTED_MANIFEST, indent=2))


In [ ]:
write_progress("Installing uv and creating eval venv")
subprocess.run(["python3", "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["python3", "-m", "uv", "venv", "--seed", "--clear", VENV_DIR], check=True)
install_commands = [
    [VENV_PYTHON, "-m", "pip", "install", "--no-cache-dir", "--upgrade", "pip", "setuptools", "wheel"],
    [VENV_PYTHON, "-m", "pip", "install", "--no-cache-dir", "numpy==1.26.4", "scipy==1.15.3", "scikit-learn==1.6.1"],
    [
        VENV_PYTHON, "-m", "pip", "install", "--no-cache-dir",
        "pillow", "packaging", "safetensors", "torchvision", "bitsandbytes", "xformers",
        "huggingface_hub>=0.34.0", "datasets==4.3.0", "transformers==4.56.2",
        "trl==0.22.2", "unsloth", "modelscope",
    ],
    [
        VENV_PYTHON, "-m", "pip", "install", "--no-cache-dir", "vllm==0.10.2",
        "--extra-index-url", "https://wheels.vllm.ai/0.10.2/",
    ],
]
for command in install_commands:
    print("Running:", " ".join(command), flush=True)
    subprocess.run(command, check=True, cwd=WORKDIR)
write_progress("Finished eval venv package installs")


In [ ]:
compat_script = """
import numpy, scipy, sklearn, torch, transformers, trl, unsloth, vllm
print({
    'numpy': numpy.__version__, 'scipy': scipy.__version__, 'sklearn': sklearn.__version__,
    'torch': torch.__version__, 'transformers': transformers.__version__, 'trl': trl.__version__, 'vllm': vllm.__version__,
})
"""
subprocess.run([VENV_PYTHON, "-c", compat_script], check=True, cwd=WORKDIR)
write_progress("Compatibility import check completed")


In [ ]:
baseline_script = r"""
import json, os, shutil, torch
from safetensors.torch import load_file, save_file

input_root = os.environ["INPUT_ROOT"]
workdir = os.environ["WORKDIR"]
template_root = os.path.join(input_root, "selected_checkpoints", "phase_a_ckpt_237")
baseline_root = os.path.join(workdir, "baseline_rank8")
if os.path.exists(baseline_root):
    shutil.rmtree(baseline_root)
os.makedirs(baseline_root, exist_ok=True)
shutil.copy(os.path.join(template_root, "adapter_config.json"), os.path.join(baseline_root, "adapter_config.json"))
tensors = load_file(os.path.join(template_root, "adapter_model.safetensors"))
zeroed = {key: torch.zeros_like(value) for key, value in tensors.items()}
save_file(zeroed, os.path.join(baseline_root, "adapter_model.safetensors"))
with open(os.path.join(baseline_root, "checkpoint_info.json"), "w", encoding="utf-8") as handle:
    json.dump({
        "phase_name": "baseline",
        "checkpoint_name": "baseline_rank8",
        "checkpoint_path": "baseline_rank8",
        "source_phase_name": "baseline",
        "notes": "Rank-8 zeroed LoRA adapter for base-model baseline evaluation.",
    }, handle, indent=2)
print("Baseline rank8 adapter:", baseline_root)
"""
env = dict(os.environ)
env["INPUT_ROOT"] = INPUT_ROOT
env["WORKDIR"] = WORKDIR
subprocess.run([VENV_PYTHON, "-c", baseline_script], check=True, env=env)
BASELINE_RANK8_DIR = os.path.join(WORKDIR, "baseline_rank8")
write_progress("Prepared baseline rank8 zero adapter")


In [ ]:
HARDWARE_PROFILE = "kaggle_t4"
EVAL_SPLIT = "testmini"
MAX_EVAL_EXAMPLES_PER_SUBSET = 4
MAX_COMPLETION_LENGTH = 64

TARGETS = [
    {"label": "baseline_rank8", "phase": "phase_a", "checkpoint": BASELINE_RANK8_DIR, "max_completion_length": MAX_COMPLETION_LENGTH},
    {"label": "phase_a_ckpt_237", "phase": "phase_a", "checkpoint": os.path.join(INPUT_ROOT, "selected_checkpoints", "phase_a_ckpt_237"), "max_completion_length": MAX_COMPLETION_LENGTH},
    {"label": "phase_b_ckpt_180", "phase": "phase_b", "checkpoint": os.path.join(INPUT_ROOT, "selected_checkpoints", "phase_b_ckpt_180"), "max_completion_length": MAX_COMPLETION_LENGTH},
    {"label": "phase_c_ckpt_240", "phase": "phase_c", "checkpoint": os.path.join(INPUT_ROOT, "selected_checkpoints", "phase_c_ckpt_240"), "max_completion_length": MAX_COMPLETION_LENGTH},
]
for target in TARGETS:
    reward_path = os.path.join(target["checkpoint"], "reward_weights.json")
    if os.path.exists(reward_path):
        target["reward_weights_json"] = reward_path
TARGET_SPEC_PATH = os.path.join(WORKDIR, "prd_final_selected_targets.json")
with open(TARGET_SPEC_PATH, "w", encoding="utf-8") as handle:
    json.dump({"targets": TARGETS}, handle, indent=2)
print(json.dumps(TARGETS, indent=2))
write_progress("Wrote selected target manifest")


In [ ]:
cmd = [
    VENV_PYTHON,
    "rl_gspo_qwen2_5vlm_eval.py",
    "--target-spec-json", TARGET_SPEC_PATH,
    "--hardware-profile", HARDWARE_PROFILE,
    "--output-root", OUTPUT_ROOT,
    "--eval-split", EVAL_SPLIT,
    "--max-eval-examples-per-subset", str(MAX_EVAL_EXAMPLES_PER_SUBSET),
    "--base-model-path", MODEL_PATH,
]
env = dict(os.environ)
env["PYTHONUNBUFFERED"] = "1"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["UNSLOTH_USE_MODELSCOPE"] = "0"
env["TRANSFORMERS_OFFLINE"] = "0"
env["HF_HUB_OFFLINE"] = "0"
env["HF_DATASETS_OFFLINE"] = "0"
reeval_log_path = os.path.join(WORKDIR, OUTPUT_ROOT, "reevaluation_subprocess.log")
os.makedirs(os.path.dirname(reeval_log_path), exist_ok=True)
print("Running:", " ".join(cmd), flush=True)
print("Subprocess log:", reeval_log_path, flush=True)
write_progress("Starting PRD final reevaluation subprocess")
with open(reeval_log_path, "w", encoding="utf-8") as log_file:
    process = subprocess.Popen(cmd, cwd=WORKDIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        log_file.write(line)
        log_file.flush()
        print(line, end="", flush=True)
    return_code = process.wait()
if return_code != 0:
    write_progress(f"PRD final reevaluation failed with return code {return_code}")
    with open(reeval_log_path, "r", encoding="utf-8") as handle:
        print("".join(handle.readlines()[-200:]), flush=True)
    raise subprocess.CalledProcessError(return_code, cmd)
write_progress("PRD final reevaluation completed successfully")


In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)

def metric_row(phase, checkpoint, metrics):
    return {
        "Phase": phase,
        "Checkpoint": checkpoint,
        "Exact": metrics.get("normalized_exact_match"),
        "Tol": metrics.get("tolerance_accuracy"),
        "Parseable": metrics.get("parseable_answer_rate"),
        "Avg Reward": metrics.get("average_total_reward"),
        "Structure": metrics.get("structure_score"),
        "Correctness": metrics.get("correctness_score"),
        "Composite": metrics.get("composite_score"),
    }

label_to_phase = {"baseline_rank8": "Baseline", "phase_a_ckpt_237": "Phase A", "phase_b_ckpt_180": "Phase B", "phase_c_ckpt_240": "Phase C"}
label_to_checkpoint = {"baseline_rank8": "baseline_rank8", "phase_a_ckpt_237": "phase_a/checkpoint-237", "phase_b_ckpt_180": "phase_b/checkpoint-180", "phase_c_ckpt_240": "phase_c/checkpoint-240"}
headline_rows=[]
stage_rows=[]
for label in ["baseline_rank8", "phase_a_ckpt_237", "phase_b_ckpt_180", "phase_c_ckpt_240"]:
    subsets = load_json(os.path.join(WORKDIR, OUTPUT_ROOT, label, "subset_metrics.json"))
    headline_rows.append(metric_row(label_to_phase[label], label_to_checkpoint[label], subsets["eval_overall_numeric"]))
    for subset in ["stage1_easy_numeric", "stage2_float_numeric", "stage3_hard_numeric"]:
        metrics = subsets.get(subset, {})
        row = metric_row(label_to_phase[label], label_to_checkpoint[label], metrics)
        row["Eval Subset"] = subset
        row["Malformed"] = metrics.get("malformed_answer_rate")
        row["Truncation"] = metrics.get("truncation_rate")
        stage_rows.append(row)

def write_csv(path, rows):
    keys=[]
    for row in rows:
        for key in row:
            if key not in keys:
                keys.append(key)
    with open(path, "w", encoding="utf-8", newline="") as handle:
        writer=csv.DictWriter(handle, fieldnames=keys)
        writer.writeheader(); writer.writerows(rows)

summary = {
    "settings": {
        "headline_eval_subset": "eval_overall_numeric",
        "eval_split": EVAL_SPLIT,
        "max_eval_examples_per_subset": MAX_EVAL_EXAMPLES_PER_SUBSET,
        "num_samples_per_prompt": 1,
        "max_completion_length": MAX_COMPLETION_LENGTH,
        "hardware_profile": HARDWARE_PROFILE,
        "lora_rank": 8,
        "kaggle_account": "KG / kgaero",
    },
    "headline_rows": headline_rows,
    "stage_wise_rows": stage_rows,
}
summary_path=os.path.join(WORKDIR, OUTPUT_ROOT, "prd_final_summary.json")
with open(summary_path, "w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)
write_csv(os.path.join(WORKDIR, OUTPUT_ROOT, "prd_headline_eval_overall_numeric.csv"), headline_rows)
write_csv(os.path.join(WORKDIR, OUTPUT_ROOT, "prd_stage_wise_diagnostics.csv"), stage_rows)
if os.path.exists(PROGRESS_PATH):
    shutil.copy(PROGRESS_PATH, os.path.join(WORKDIR, OUTPUT_ROOT, "progress.txt"))
print("PRD final summary:")
print(json.dumps(summary, indent=2))
write_progress("PRD final summary artifacts written")
